# Hierarchical & Density-Based Clustering

**Companion lesson:** https://ml-viz.vercel.app/courses/clustering/02-hierarchical-and-dbscan

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## DBSCAN: Finding Arbitrary Shapes

Core points have `min_samples` neighbors within `eps`.
Border points are near core points.
Noise points are neither.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, eps in zip(axes, [0.1, 0.2, 0.5]):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    unique_labels = set(labels)
    colors = ['#818cf8', '#14b8a6', '#eab308', '#f43f5e']
    for label in unique_labels:
        mask = labels == label
        if label == -1:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c='#f43f5e', s=20, marker='x', alpha=0.7, label='Noise')
        else:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c=colors[label % len(colors)], s=15, alpha=0.7)
    ax.set_title(f'eps={eps}\n{n_clusters} clusters, {n_noise} noise pts', color='white', fontsize=11)
    ax.set_xlim(-2, 3)
    ax.set_ylim(-1.5, 2)
    ax.set_aspect('equal')
plt.suptitle('DBSCAN: eps Controls Cluster Discovery', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN by hand: core / border / noise on the 9-point example
# (matches the lesson). KEY: min_samples counts the point ITSELF.
import numpy as np
from sklearn.cluster import DBSCAN

pts = np.array([[1,1],[1,2],[2,1],[2,2],[8,8],[8,9],[9,8],[9,9],[5,5]])
eps, min_samples = 2.0, 3

D = np.linalg.norm(pts[:, None] - pts[None], axis=2)   # pairwise distances
neigh_counts = (D <= eps).sum(axis=1)                  # includes self (D=0 <= eps)
is_core = neigh_counts >= min_samples

print(f'eps={eps}, min_samples={min_samples}  (a core point needs {min_samples-1} OTHER neighbors)\n')
for i, p in enumerate(pts):
    others = [tuple(pts[j]) for j in range(len(pts)) if j != i and D[i, j] <= eps]
    kind = 'CORE' if is_core[i] else ('noise' if not others else 'border')
    print(f'  {tuple(p)}: |N_eps| incl. self = {neigh_counts[i]}  neighbors={others}  -> {kind}')

# Cross-check against scikit-learn
labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(pts)
print(f'\nsklearn labels: {labels}   (-1 = noise)')
print('(5,5) is noise:', labels[-1] == -1)


## Hierarchical Clustering

Build a dendrogram by merging the closest clusters step by step.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
n = 50
X_small = np.vstack([np.random.randn(20, 2) + [0, 0],
                     np.random.randn(15, 2) + [4, 0],
                     np.random.randn(15, 2) + [2, 4]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_small[:, 0], X_small[:, 1], c='#818cf8', s=15, alpha=0.7)
axes[0].set_title('Data Points', color='white')
axes[0].set_aspect('equal')

Z = linkage(X_small, method='ward')
dendrogram(Z, ax=axes[1], color_threshold=7, leaf_font_size=8)
axes[1].set_title('Ward Linkage Dendrogram', color='white')
axes[1].axhline(y=7, color='#f43f5e', linestyle='--', alpha=0.7, label='Cut → 3 clusters')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Linkage criteria

Agglomerative clustering merges the closest clusters; the **linkage** defines 'closest' (single = nearest pair, complete = farthest pair, ward = min variance increase).

In [ ]:
# Hierarchical by hand: A,B,C,D — distance matrix + per-linkage final-merge height
import numpy as np
from scipy.cluster.hierarchy import linkage

P = {'A': (1,1), 'B': (2,2), 'C': (8,8), 'D': (9,9)}
names = list(P); X4 = np.array(list(P.values()))
D = np.linalg.norm(X4[:, None] - X4[None], axis=2)
print('Distance matrix:')
print('     ' + '   '.join(names))
for i, n in enumerate(names):
    print(f'  {n}  ' + '  '.join(f'{D[i,j]:5.2f}' for j in range(4)))

# Merges 1 & 2: A+B and C+D both at 1.41 (smallest off-diagonal entries)
AB, CD = [0, 1], [2, 3]
cross = D[np.ix_(AB, CD)].ravel()    # the 4 A/B-to-C/D distances
print('\nAfter merging A+B and C+D (both at height 1.41), the {AB}-{CD} cross distances:')
print('  ', np.round(cross, 2).tolist())
print(f'  single  (min)  = {cross.min():.2f}')
print(f'  complete(max)  = {cross.max():.2f}')
print(f'  average (mean) = {cross.mean():.2f}')

# Cross-check the final-merge height that scipy reports for each linkage
print('\nscipy linkage final-merge height:')
for m in ['single', 'complete', 'average']:
    Z = linkage(X4, method=m)
    print(f'  {m:9s}: {Z[-1, 2]:.2f}')


In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=120, centers=3, cluster_std=0.7, random_state=0)
for method in ['single', 'complete', 'average', 'ward']:
    Z = linkage(X, method=method)
    labels = fcluster(Z, t=3, criterion='maxclust')
    print(f'{method:9s}: {len(np.unique(labels))} clusters, sizes {np.bincount(labels)[1:]}')

## Key takeaways

- **Hierarchical** clustering builds a dendrogram; cut it at any level for K clusters — no K upfront.
- **Linkage** (single/complete/average/ward) changes cluster shape; ward favors compact clusters.
- **DBSCAN** finds arbitrary shapes by density and labels sparse points as **noise** — no K needed.
- Tune DBSCAN's `eps` via the k-distance elbow; it struggles with varying densities.